# 07 - Forest plots

One forest plot per variant of interest: the per-study effect sizes with the
fixed- and random-effect summaries from PLINK's `.meta` output.

Per-study betas are aligned to the meta effect allele (sign flipped where A1
differs), studies with an implausible SE are shown but excluded, and the
summary CIs are derived from PLINK's p-values so the drawn interval and the
printed p-value cannot disagree.

In [ ]:
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd

import gwas_config as cfg
import gwas_forest as forest
import gwas_meta as meta

pd.options.mode.chained_assignment = None

cfg.describe()

In [ ]:
# Mount the release and working buckets with gcsfuse (run once per VM).
for cmd in cfg.mount_commands():
    print(cmd)

## Variants of interest

In [ ]:
# Curated table of variants
curated_path = cfg.prep_path('meta', 'forest_plots', 'curated_variants.csv')
curated = pd.read_csv(curated_path) if curated_path.exists() else None
if curated is not None:
    display(curated)

# Or explicit list of chrom:pos:ref:alt IDs
TARGET_SNPS = [
    'chr13:47347930:T:C', 'chr9:72900893:C:G', 'chr1:3191409:C:T',       # SURV
    'chr12:109458542:C:T', 'chr7:19172892:C:G', 'chr18:26909098:G:A',    # CS_disc
]
print(f'{len(TARGET_SNPS)} target variants')

## Per-study results and meta tables

The same collection used by the meta-analysis, so the plots always show the
studies that actually went into it.

In [ ]:
ANCESTRIES = ['EUR', 'AJ']

study_files = {
    analysis: meta.labelled_paths(
        meta.collect_result_paths(analysis, ancestries=ANCESTRIES, verbose=False))
    for analysis in ('CS_disc', 'SURV')
}

for analysis, files in study_files.items():
    print(f'{analysis}: {len(files)} studies')
    for name in files:
        print(f'    {name}')

In [ ]:
# Read only the target variants out of each study.
study_data = {
    analysis: {name: forest.load_study_subset(path, TARGET_SNPS)
               for name, path in files.items()}
    for analysis, files in study_files.items()
}

for analysis, data in study_data.items():
    print(f'{analysis}: ' + ', '.join(f'{s}={len(d)}' for s, d in data.items()))

In [ ]:
# PLINK .meta tables written by notebook 05, indexed by SNP.
META_LABEL = {
    'CS_disc': f'CS_disc_{"_".join(ANCESTRIES)}',
    'SURV': f'SURV_{"_".join(ANCESTRIES)}',
}

meta_tables = {}
for analysis, label in META_LABEL.items():
    path = cfg.prep_path('meta', 'PLINK', analysis, f'{label}.meta')
    table = pd.read_csv(path, sep=r'\s+')
    meta_tables[analysis] = table[table['SNP'].isin(TARGET_SNPS)].set_index('SNP')
    print(f'{analysis}: {len(meta_tables[analysis])} of {len(TARGET_SNPS)} targets '
          'have a meta result')

## Plot

`beta_label` is log(OR) for the cross-sectional model and log(HR) for the
survival model.

In [ ]:
out_root = cfg.prep_path('meta', 'forest_plots')
BETA_LABELS = {'CS_disc': 'Beta (log OR)', 'SURV': 'Beta (log HR)'}

summaries = {}
for analysis in ('CS_disc', 'SURV'):
    out_dir = out_root / analysis
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for snp in TARGET_SNPS:
        res = forest.forest_plot(
            snp, study_data[analysis], meta_tables[analysis],
            beta_label=BETA_LABELS[analysis],
            save_path=out_dir / f'{snp.replace(":", "_")}.png')
        if res is not None:
            rows.append(res)

    summaries[analysis] = pd.DataFrame(rows)
    summaries[analysis].to_csv(out_root / f'{analysis}_meta_summary.csv', index=False)
    print(f'{analysis}: {len(rows)} plots -> {out_dir}')

display(summaries['CS_disc'])
display(summaries['SURV'])

## Studies with implausible standard errors

A huge SE means the model did not converge for that study and variant; those
rows are excluded from the plot, and are worth listing so the cause can be
chased down.

In [ ]:
SE_THRESH = 100

for analysis, data in study_data.items():
    extreme = forest.extreme_studies(data, se_thresh=SE_THRESH)
    print(f'{analysis}: {len(extreme)} rows with |SE| > {SE_THRESH}'
          f' over {extreme["SNP"].nunique() if len(extreme) else 0} variants')
    if len(extreme):
        extreme.to_csv(out_root / f'{analysis}_extreme_studies.csv', index=False)
        display(extreme.groupby('SNP').size().sort_values(ascending=False).head(20))

## A locus with no meta result

When a variant is missing from the meta-analysis plots the per-study estimates alone.

In [ ]:
LOCUS = 'chr1:155235843'

locus_data = {
    analysis: {name: forest.load_locus(path, f'{LOCUS}:')
               for name, path in files.items()}
    for analysis, files in study_files.items()
}

found = sorted({i for data in locus_data.values()
                for d in data.values() for i in d['ID'].unique()})
print('SNP IDs found:', found)

In [ ]:
snp_id = found[0]
out_dir = out_root / 'no_meta'
out_dir.mkdir(parents=True, exist_ok=True)

for analysis in ('CS_disc', 'SURV'):
    forest.forest_plot(
        snp_id, locus_data[analysis], meta_df=None,
        beta_label=BETA_LABELS[analysis],
        save_path=out_dir / f'{snp_id.replace(":", "_")}_{analysis}.png')
print(f'wrote plots to {out_dir}')

## Per-cohort breakdown of a pooled study

A pooled study (e.g. LR GP2 EUR cohorts) can be opened up by running the same
model per cohort and drawing those as indented rows under their parent. The
breakdown rows are display-only: they are not part of the meta-analysis and do
not affect k or the x-axis scale.

In [ ]:
# Per-cohort runs, e.g. from a focus run stratified by cohort with the same
# covariates minus `cohort` itself.
COHORT_RES_DIR = Path(os.environ.get('GWAS_COHORT_RUN_DIR', ''))
SUB_PARENT = 'EUR_case_control'    # the pooled study these cohorts make up
ANALYSIS = 'CS_disc'

cohort_files = {Path(f).name.replace('_step07_final.y.results', ''): f
                for f in sorted(glob.glob(f'{COHORT_RES_DIR}/*_step07_final.y.results'))}
print(f'{len(cohort_files)} per-cohort result files')

cohort_data = {c: forest.load_study_subset(f, TARGET_SNPS)
               for c, f in cohort_files.items()}
bd_snps = sorted(set().union(*(set(d['ID']) for d in cohort_data.values())))
print(f'{len(bd_snps)} target variants have a cohort breakdown')

In [ ]:
bd_dir = out_root / f'{ANALYSIS}_cohorts'
bd_dir.mkdir(parents=True, exist_ok=True)

n = 0
for snp in bd_snps:
    res = forest.forest_plot(
        snp, study_data[ANALYSIS], meta_tables[ANALYSIS],
        beta_label=BETA_LABELS[ANALYSIS],
        save_path=bd_dir / f'{snp.replace(":", "_")}.png',
        sub_data=cohort_data, sub_parent=SUB_PARENT)
    n += res is not None
print(f'saved {n} plots to {bd_dir}')